# EnterYourTeamName - Task 2 inference

Reproducible `d2-e002-metarank` inference notebook. It runs unchanged in a local VS Code/Jupyter session or in Kaggle and writes `submission.csv`. No external data, network call, API, or pretrained weight is used.

In [ ]:
import hashlib
import json
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

started = time.perf_counter()
EXPERIMENT_ID = 'd2-e002-metarank'
EXPECTED_PREDICTION_SHA256 = '292bb1567ac81cd70b87b1f4730468830640388919b72126aef69f198e9d1ba0'
EXPECTED_CSV_SHA256 = '87b4a480008eabe06921a36edc14ea12abf4a006b86850f5260d959c54ad3d81'
REQUIRED_FILES = {
    'states_train.csv',
    'states_test.csv',
    'articles.csv',
    'categories.csv',
    'sample_submission.csv',
}


In [ ]:
def is_data_root(path):
    path = Path(path)
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_FILES)


def find_data_root():
    override = os.environ.get('TASK2_DATA_DIR')
    if override:
        root = Path(override).expanduser().resolve()
        if not is_data_root(root):
            raise FileNotFoundError(f'TASK2_DATA_DIR is not a valid dataset root: {root}')
        return root

    cwd = Path.cwd().resolve()
    local_candidates = [
        cwd / 'task2' / 'data' / 'competition' / 'dataset-task2',
        cwd / 'data' / 'competition' / 'dataset-task2',
        cwd / 'dataset-task2',
    ]
    for ancestor in [cwd, *cwd.parents]:
        local_candidates.append(
            ancestor / 'task2' / 'data' / 'competition' / 'dataset-task2'
        )
    local_valid = []
    for candidate in local_candidates:
        if is_data_root(candidate):
            resolved = candidate.resolve()
            if resolved not in local_valid:
                local_valid.append(resolved)
    if len(local_valid) == 1:
        return local_valid[0]
    if len(local_valid) > 1:
        raise RuntimeError(f'Multiple local Task 2 datasets found: {local_valid}')

    kaggle_input = Path('/kaggle/input')
    kaggle_valid = sorted({
        path.parent.resolve()
        for path in kaggle_input.rglob('states_train.csv')
        if is_data_root(path.parent)
    }) if kaggle_input.is_dir() else []
    if len(kaggle_valid) != 1:
        raise FileNotFoundError(
            'Expected exactly one Task 2 dataset root. '
            f'Found local={local_valid}, kaggle={kaggle_valid}'
        )
    return kaggle_valid[0]


def find_output_path():
    override = os.environ.get('TASK2_SUBMISSION_PATH')
    if override:
        return Path(override).expanduser().resolve()
    kaggle_working = Path('/kaggle/working')
    if kaggle_working.is_dir():
        return kaggle_working / 'submission.csv'
    cwd = Path.cwd().resolve()
    for ancestor in [cwd, *cwd.parents]:
        if (ancestor / 'task2').is_dir():
            return ancestor / 'task2' / 'submissions' / 'submission.csv'
    return cwd / 'submission.csv'


data_root = find_data_root()
output_path = find_output_path()
print(f'data_root={data_root}')
print(f'output_path={output_path}')


In [ ]:
train = pd.read_csv(data_root / 'states_train.csv')
test = pd.read_csv(data_root / 'states_test.csv')
articles = pd.read_csv(data_root / 'articles.csv')
categories = pd.read_csv(data_root / 'categories.csv')
sample = pd.read_csv(data_root / 'sample_submission.csv')

expected_train_columns = [
    'state_id', 'current_article_id', 'target_article_id', 'next_article_id'
]
expected_test_columns = ['state_id', 'current_article_id', 'target_article_id']
expected_submission_columns = ['state_id', 'predicted_next_article_id']
if train.columns.tolist() != expected_train_columns:
    raise ValueError(f'Unexpected train columns: {train.columns.tolist()}')
if test.columns.tolist() != expected_test_columns:
    raise ValueError(f'Unexpected test columns: {test.columns.tolist()}')
if sample.columns.tolist() != expected_submission_columns:
    raise ValueError(f'Unexpected sample columns: {sample.columns.tolist()}')
if articles.columns.tolist() != ['article_id', 'title']:
    raise ValueError(f'Unexpected article columns: {articles.columns.tolist()}')
if categories.columns.tolist() != ['article_id', 'category']:
    raise ValueError(f'Unexpected category columns: {categories.columns.tolist()}')
if len(test) != len(sample) or not sample['state_id'].equals(test['state_id']):
    raise ValueError('Test and sample state IDs differ in rows or order')
if any(frame.isna().any().any() for frame in (train, test, articles, categories)):
    raise ValueError('Official inputs contain missing values')

print({
    'train_rows': len(train),
    'test_rows': len(test),
    'articles': len(articles),
    'train_targets': train['target_article_id'].nunique(),
    'test_targets': test['target_article_id'].nunique(),
})


In [ ]:
TITLE_TOKEN_PATTERN = re.compile(r'[a-z0-9]+')


def title_tokens(title):
    return frozenset(TITLE_TOKEN_PATTERN.findall(str(title).casefold()))


def jaccard(left, right):
    union = left | right
    return len(left & right) / len(union) if union else 0.0


def deterministic_mode(values):
    counts = values.astype(np.int64).value_counts(sort=False)
    maximum = int(counts.max())
    return int(min(int(value) for value in counts[counts == maximum].index))


title_tokens_by_article = {
    int(row.article_id): title_tokens(row.title)
    for row in articles.itertuples(index=False)
}
categories_by_article = {
    int(article_id): frozenset(group['category'].astype(str))
    for article_id, group in categories.groupby('article_id', sort=True)
}
candidate_counts_by_current = {
    int(current): Counter(group['next_article_id'].astype(np.int64))
    for current, group in train.groupby('current_article_id', sort=True)
}
global_next_counts = Counter(train['next_article_id'].astype(np.int64))
global_fallback = deterministic_mode(train['next_article_id'])


def semantic_score(candidate, target):
    return 2.0 * jaccard(
        categories_by_article.get(candidate, frozenset()),
        categories_by_article.get(target, frozenset()),
    ) + jaccard(
        title_tokens_by_article.get(candidate, frozenset()),
        title_tokens_by_article.get(target, frozenset()),
    )


def predict_one(current, target):
    counts = candidate_counts_by_current.get(current)
    if not counts:
        return global_fallback
    return int(max(
        counts,
        key=lambda candidate: (
            semantic_score(int(candidate), target),
            counts[int(candidate)],
            global_next_counts[int(candidate)],
            -int(candidate),
        ),
    ))


prediction = np.fromiter(
    (predict_one(int(current), int(target))
     for current, target in zip(
         test['current_article_id'], test['target_article_id'], strict=True
     )),
    dtype=np.int64,
    count=len(test),
)
submission = sample.copy()
submission['predicted_next_article_id'] = prediction


In [ ]:
if submission.columns.tolist() != expected_submission_columns:
    raise ValueError('Submission columns changed unexpectedly')
if len(submission) != len(sample):
    raise ValueError('Submission row count does not match sample')
if not submission['state_id'].equals(sample['state_id']):
    raise ValueError('Submission state IDs or order do not match sample')
if submission['state_id'].duplicated().any():
    raise ValueError('Submission state IDs are not unique')
if not np.isfinite(prediction).all():
    raise ValueError('Predictions contain non-finite values')
article_universe = set(articles['article_id'].astype(np.int64))
invalid = [int(value) for value in prediction if int(value) not in article_universe]
if invalid:
    raise ValueError(f'Predictions outside article universe: {invalid[:10]}')
prediction_sha256 = hashlib.sha256(
    prediction.astype('<i8', copy=False).tobytes()
).hexdigest()
if prediction_sha256 != EXPECTED_PREDICTION_SHA256:
    raise RuntimeError(
        f'Prediction hash mismatch: {prediction_sha256} != {EXPECTED_PREDICTION_SHA256}'
    )

output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)
csv_sha256 = hashlib.sha256(output_path.read_bytes()).hexdigest()
if csv_sha256 != EXPECTED_CSV_SHA256:
    raise RuntimeError(f'CSV hash mismatch: {csv_sha256} != {EXPECTED_CSV_SHA256}')
summary = {
    'experiment_id': EXPERIMENT_ID,
    'rows': int(len(submission)),
    'unique_state_ids': int(submission['state_id'].nunique()),
    'unique_predictions': int(submission['predicted_next_article_id'].nunique()),
    'seen_current_rows': int(test['current_article_id'].isin(candidate_counts_by_current).sum()),
    'global_fallback_article_id': global_fallback,
    'prediction_sha256': prediction_sha256,
    'csv_sha256': csv_sha256,
    'runtime_seconds': float(time.perf_counter() - started),
    'output': str(output_path),
    'status': 'READY',
}
print(json.dumps(summary, indent=2))
